# Sentiment Analysis on IMDB Dataset 

In [ ]:
from functools import partial
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch import optim

import torchtext.datasets as datasets
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

print(f"PyTorch version: {torch.__version__}")

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

In [ ]:
print("Loading IMDB dataset...")
train_iter = datasets.IMDB(split='train')
train_data = list(train_iter)
print(f"Train data loaded: {len(train_data)} samples")

In [ ]:
eval_iter = datasets.IMDB(split='test')
eval_data = list(eval_iter)
print(f"Test data loaded: {len(eval_data)} samples")

In [ ]:
# Check data structure
print("Sample data point:")
label, review = train_data[0]
print(f"Label: {label}")
print(f"Review (first 200 chars): {review[:200]}...")
print(f"\nLabel type: {type(label)}")
print(f"Review type: {type(review)}")

In [ ]:
tokenizer = get_tokenizer("basic_english")

In [ ]:
min_word_freq = 2

def build_vocab(train_data, tokenizer):
    reviews = [review for label, review in train_data]
    vocab = build_vocab_from_iterator(
        map(tokenizer, reviews),
        specials=["<unk>", "<eos>", "<pad>"],
        min_freq=min_word_freq
    )
    vocab.set_default_index(vocab["<unk>"])
    return vocab

In [ ]:
print("Building vocabulary...")
vocab = build_vocab(train_data, tokenizer)
print("Vocabulary built successfully")

In [ ]:
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

In [ ]:
max_seq_len = 256
max_norm = 1
embed_dim = 300
batch_size = 16

text_pipeline = lambda x: vocab(tokenizer(x))

In [ ]:
sample = text_pipeline("Hello World")
print(f"Sample tokens: {sample}")
print(f"Type: {type(sample)}")

In [ ]:
print(f"Special tokens: {vocab(['<unk>', '<eos>', '<pad>'])}")

In [ ]:
def collate_data(batch, text_pipeline):
    reviews, targets = [], []
    
    for label, review in batch:
        review_tokens_ids = text_pipeline(review)
        
        if max_seq_len:
            review_tokens_ids = review_tokens_ids[:max_seq_len]
        
        review_tokens_ids.append(1)  # <eos>
        l = len(review_tokens_ids)
        
        x = [2] * 257  # <pad>
        x[:l] = review_tokens_ids
        
        reviews.append(x)
        targets.append(label)
    
    reviews = torch.tensor(reviews, dtype=torch.long)
    targets = torch.tensor(targets, dtype=torch.long)
    
    return reviews, targets

In [ ]:
traindl = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=partial(collate_data, text_pipeline=text_pipeline)
)

evaldl = DataLoader(
    eval_data,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=partial(collate_data, text_pipeline=text_pipeline)
)

In [ ]:
# Test dataloader
for i, (reviews, labels) in enumerate(traindl):
    print(f"Reviews shape: {reviews.shape}")
    print(f"Labels shape: {labels.shape}")
    break

In [ ]:
class SentiNN(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size):
        super().__init__()
        self.e = nn.Embedding(input_size, embed_size)
        self.dropout = nn.Dropout(0.2)
        self.rnn = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.out = nn.Linear(in_features=hidden_size, out_features=2)
    
    def forward(self, x):
        x = self.e(x)
        x = self.dropout(x)
        outputs, hidden = self.rnn(x)
        hidden.squeeze_(0)
        logits = self.out(hidden)
        return logits

In [ ]:
embed_size = 128
hidden_size = 256

sentinn = SentiNN(vocab_size, embed_size, hidden_size).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=2).to(device)
lr = 0.001
opt = optim.Adam(params=sentinn.parameters(), lr=lr)

print(f"Model initialized on {device}")
print(f"Vocabulary size: {vocab_size}")
print(f"Embedding size: {embed_size}")
print(f"Hidden size: {hidden_size}")

In [ ]:
def train_one_epoch():
    sentinn.train()
    track_loss = 0
    num_correct = 0
    
    for i, (reviews_ids, sentiments) in enumerate(traindl):
        reviews_ids = reviews_ids.to(device)
        sentiments = sentiments.to(device) - 1
        
        logits = sentinn(reviews_ids)
        loss = loss_fn(logits, sentiments)
        
        track_loss += loss.item()
        num_correct += (torch.argmax(logits, dim=1) == sentiments).type(torch.float).sum().item()
        
        running_loss = round(track_loss / (i + (reviews_ids.shape[0] / batch_size)), 4)
        running_acc = round((num_correct / ((i * batch_size + reviews_ids.shape[0]))) * 100, 4)
        
        opt.zero_grad()
        loss.backward()
        opt.step()
    
    epoch_loss = running_loss
    epoch_acc = running_acc
    return epoch_loss, epoch_acc

In [ ]:
def eval_one_epoch():
    sentinn.eval()
    track_loss = 0
    num_correct = 0
    
    with torch.no_grad():
        for i, (reviews_ids, sentiments) in enumerate(evaldl):
            reviews_ids = reviews_ids.to(device)
            sentiments = sentiments.to(device) - 1
            
            logits = sentinn(reviews_ids)
            loss = loss_fn(logits, sentiments)
            
            track_loss += loss.item()
            num_correct += (torch.argmax(logits, dim=1) == sentiments).type(torch.float).sum().item()
            
            running_loss = round(track_loss / (i + (reviews_ids.shape[0] / batch_size)), 4)
            running_acc = round((num_correct / ((i * batch_size + reviews_ids.shape[0]))) * 100, 4)
    
    epoch_loss = running_loss
    epoch_acc = running_acc
    return epoch_loss, epoch_acc

In [ ]:
n_epochs = 10

print("Starting training...\n")
for e in range(n_epochs):
    print(f"Epoch {e+1}/{n_epochs}", end=", ")
    epoch_loss, epoch_acc = train_one_epoch()
    print(f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%", end=", ")
    epoch_loss, epoch_acc = eval_one_epoch()
    print(f"Eval Loss: {epoch_loss:.4f}, Eval Acc: {epoch_acc:.2f}%")

## Notes

Scope for improvement:
- Better network architecture
- Measures to address overfitting
- Better handling of variable length sequences
- Improved training strategies